# Pokemon Battle Prediction - Data Preprocessing

## Project Overview
This notebook demonstrates the preprocessing pipeline for a Pokemon battle outcome prediction model using the [Pokedex Pokemon Data](https://www.kaggle.com/datasets/lmno3418/pokedex-pokemon-data) dataset from Kaggle.

## Objective
To clean and transform raw Pokemon battle data into features suitable for machine learning models that can predict battle outcomes.

## Dataset Description
The dataset contains Pokemon battle data with various attributes including:
- Pokemon stats (HP, Attack, Defense, etc.)
- Pokemon types and legendary status
- Physical attributes (height, weight)
- Battle outcomes

---

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Initial Exploration

In [2]:
train_df=pd.read_csv('train.csv')
train_df

,First_pokemon_id,Second_pokemon_id,Winner,Name_first,Type 1_first,Type 2_first,HP_first,Attack_first,Defense_first,Sp. Atk_first,...,Attack_second,Defense_second,Sp. Atk_second,Sp. Def_second,Speed_second,Generation_second,Legendary_second,height_second,weight_second,base_experience_second
0,266,298,298,Larvitar,Rock,Ground,50,64,50,45,...,70,40,60,40,60,3,False,10,280,119
1,702,701,701,Virizion,Grass,Fighting,91,90,72,90,...,129,90,72,90,108,5,True,19,2600,261
2,191,668,668,Togetic,Fairy,Flying,55,40,85,80,...,75,75,125,95,40,5,False,10,345,170
3,237,683,683,Slugma,Fire,Normal,40,40,40,70,...,120,90,60,90,48,5,False,16,1390,170
4,151,231,151,Omastar,Rock,Water,70,60,125,115,...,10,230,10,230,5,2,False,6,205,177
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
38049,657,681,681,Joltik,Bug,Electric,50,47,50,57,...,85,50,55,50,65,5,False,9,200,70
38050,707,126,707,Reshiram,Dragon,Fire,100,120,100,150,...,40,70,70,25,60,1,False,4,80,59
38051,589,664,589,Drilbur,Ground,Normal,60,85,40,30,...,55,40,45,40,60,5,False,2,3,55
38052,303,368,368,Pelipper,Water,Flying,60,50,100,85,...,115,60,60,60,90,3,False,13,403,160


### Dataset Information

In [3]:
print(train_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 38054 entries, 0 to 38053
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   First_pokemon_id        38054 non-null  int64 
 1   Second_pokemon_id       38054 non-null  int64 
 2   Winner                  38054 non-null  int64 
 3   Name_first              38054 non-null  object
 4   Type 1_first            38054 non-null  object
 5   Type 2_first            38054 non-null  object
 6   HP_first                38054 non-null  int64 
 7   Attack_first            38054 non-null  int64 
 8   Defense_first           38054 non-null  int64 
 9   Sp. Atk_first           38054 non-null  int64 
 10  Sp. Def_first           38054 non-null  int64 
 11  Speed_first             38054 non-null  int64 
 12  Generation_first        38054 non-null  int64 
 13  Legendary_first         38054 non-null  bool  
 14  height_first            38054 non-null  int64 
 15  we

**Key Observations:**
- The dataset contains battle records between pairs of Pokemon
- Each row represents one battle with two Pokemon and their respective stats
- We need to engineer features that capture the relative differences between Pokemon

---


## Feature Engineering

### 1. Target Variable Creation
We create a binary target variable to indicate which Pokemon won the battle:

In [4]:
train_df['first_winner'] = (train_df['First_pokemon_id'] == train_df['Winner']).astype(int)
train_df['second_winner'] = (train_df['Second_pokemon_id'] == train_df['Winner']).astype(int)

### 2. Data Cleanup
Remove unnecessary columns that won't be used for modeling. Also removed the second_winner variable as the first_winner variable is emough to know which pokemon won. 1 means first pokemon won 0 means second pokemon won.

In [5]:
columns_to_drop=['Name_first','Name_second','First_pokemon_id','Second_pokemon_id','Winner','second_winner']
train_df = train_df.drop(columns=columns_to_drop)

### 3. Categorical Variable Encoding
Convert boolean legendary status to numeric:

In [6]:
train_df['Legendary_first'] = train_df['Legendary_first'].astype(int)
train_df['Legendary_second'] = train_df['Legendary_second'].astype(int)

---


## Statistical Difference Features

The key insight is that battle outcomes likely depend on the **differences** between Pokemon stats rather than absolute values. We create difference features for all numerical attributes:

In [7]:
train_df['HP_diff'] = train_df['HP_first'] - train_df['HP_second']
train_df['Attack_diff'] = train_df['Attack_first'] - train_df['Attack_second']
train_df['Defense_diff'] = train_df['Defense_first'] - train_df['Defense_second']
train_df['Sp_Atk_diff'] = train_df['Sp. Atk_first'] - train_df['Sp. Atk_second']
train_df['Sp_Def_diff'] = train_df['Sp. Def_first'] - train_df['Sp. Def_second']
train_df['Speed_diff'] = train_df['Speed_first'] - train_df['Speed_second']
train_df['Height_diff'] = train_df['height_first'] - train_df['height_second']
train_df['Weight_diff'] = train_df['weight_first'] - train_df['weight_second']
train_df['Experience_diff'] = train_df['base_experience_first'] - train_df['base_experience_second']

**Rationale:** 
- Positive difference = First Pokemon has advantage
- Negative difference = Second Pokemon has advantage
- Zero difference = Equal stats

### Remove Original Stat Columns

In [8]:
columns_to_drop=['HP_first','Attack_first','Defense_first','Sp. Atk_first','Sp. Def_first','Speed_first','Generation_first','height_first','weight_first'
                ,'base_experience_first','HP_second','Attack_second','Defense_second','Sp. Atk_second','Sp. Def_second','Speed_second','Generation_second','height_second','weight_second'
                ,'base_experience_second']
train_df = train_df.drop(columns=columns_to_drop)

---

## Pokemon Type Encoding

Pokemon types play a crucial role in battle outcomes due to type effectiveness. We use one-hot encoding for all Pokemon types as it is easier for a model to understand numeric values than the types as names:


In [9]:
types = ['Normal', 'Fire', 'Water', 'Grass', 'Flying', 'Fighting', 'Poison', 'Electric',
         'Ground', 'Rock', 'Psychic', 'Ice', 'Bug', 'Ghost', 'Steel', 'Dragon', 'Dark', 'Fairy']

for t in types:
    train_df[f'{t}_first'] = ((train_df['Type 1_first'] == t) | (train_df['Type 2_first'] == t)).astype(int)

# For the second Pokémon
for t in types:
    train_df[f'{t}_second'] = ((train_df['Type 1_second'] == t) | (train_df['Type 2_second'] == t)).astype(int)

**Note:** This encoding captures both primary and secondary types for each Pokemon.

### Remove Original Type Columns

In [10]:
columns_to_drop=['Type 1_first','Type 2_first','Type 1_second','Type 2_second']
train_df = train_df.drop(columns=columns_to_drop)

---

## Speed-Based Attack Order Features

In Pokemon battles, the faster Pokemon typically attacks first. We create features to capture this battle mechanic:

In [11]:
train_df['First_pokemon_attacks_first'] = train_df['Speed_diff'].apply(lambda x: 1 if x >= 0 else 0)
train_df['Second_pokemon_attacks_first'] = train_df['Speed_diff'].apply(lambda x: 1 if x <= 0 else 0)
columns_to_drop=['Speed_diff']
train_df = train_df.drop(columns=columns_to_drop)

**Strategic Insight:** Going first in Pokemon battles provides a significant advantage, making this an important feature.

---

## Feature Scaling

Normalize the difference features to ensure all features contribute equally to the model:

In [12]:
from sklearn.preprocessing import MinMaxScaler

In [13]:
diff_columns = [
    'HP_diff','Attack_diff', 'Defense_diff', 'Sp_Atk_diff', 'Sp_Def_diff',
    'Height_diff', 'Weight_diff', 'Experience_diff'
]

scaler = MinMaxScaler()

train_df[diff_columns] = scaler.fit_transform(train_df[diff_columns])

print(train_df[diff_columns].head())

    HP_diff  Attack_diff  Defense_diff  Sp_Atk_diff  Sp_Def_diff  Height_diff  \
0  0.455556     0.455738      0.533175     0.440141     0.521845     0.484211   
1  0.500000     0.347541      0.466825     0.556338     0.592233     0.501754   
2  0.455556     0.360656      0.533175     0.334507     0.521845     0.484211   
3  0.417778     0.213115      0.390995     0.528169     0.376214     0.466667   
4  0.611111     0.639344      0.260664     0.862676     0.109223     0.512281   

   Weight_diff  Experience_diff  
0     0.523408         0.447392  
1     0.468640         0.499558  
2     0.483754         0.474801  
3     0.445468         0.393457  
4     0.507873         0.496021  


**Why MinMax Scaling?**
- Ensures all features are on the same scale (0-1)
- Prevents features with larger ranges from dominating the model
- Maintains the relative relationships between data points

---

In [14]:
train_df.to_csv('pre-processed-dataset.csv', index=False)

## Final Dataset Summary

The preprocessing pipeline transforms the raw Pokemon battle data into a machine learning-ready format with:

**Target Variable:**
- `first_winner`: Binary indicator (1 = first Pokemon wins, 0 = second Pokemon wins)

**Feature Categories:**
1. **Stat Differences** (8 features): HP, Attack, Defense, Sp. Attack, Sp. Defense, Height, Weight, Experience
2. **Type Encoding** (36 features): One-hot encoded types for both Pokemon
3. **Legendary Status** (2 features): Binary indicators for legendary Pokemon
4. **Battle Mechanics** (2 features): Attack order based on speed

**Total Features:** 48 engineered features optimized for Pokemon battle prediction

The preprocessed dataset is now ready for machine learning model training and evaluation.